# Tree Algorithms

Notes and runnable examples on the classic algorithms that run **on** a tree — rooting it, finding lowest common ancestors, measuring its diameter, and computing an answer for *every* possible root in one sweep. The structure itself (nodes, traversals, BST balancing) lives in the **trees** notebook; here we treat a tree as the special graph it is — *connected, acyclic, n nodes and exactly n−1 edges* — and exploit that structure for algorithms a general **graph** can't match.

The throughline: a tree has **no cycles**, so a single DFS from a root fixes a parent, a depth, and a subtree for every node — and almost every tree algorithm is just bookkeeping layered on that one pass. Every nontrivial claim below is checked against a brute-force oracle over many seeded-random trees.

**Contents**

1. **A tree is a special graph** — root it; parent / depth / subtree-size in one DFS
2. **Lowest common ancestor** — binary lifting (a sparse table of ancestors)
3. **Tree diameter** — two-pass BFS *and* the single-DFS DP
4. **Rerooting** — an answer for every root in O(n)
5. **When to use what**
6. **Complexity cheat-sheet**


## 1. A tree is a special graph

A **tree** is an undirected graph that is **connected** and **acyclic** — equivalently, *n* nodes joined by exactly *n − 1* edges, with a unique path between any two nodes. That last property is the gold: no cycles means no "did I already visit this?" ambiguity beyond not walking back to your parent.

We store it exactly like any graph (an adjacency list — see the **graphs** notebook), but the moment we pick a **root**, a single DFS hands us three arrays that power everything downstream:

- **`parent[v]`** — the node one step toward the root (`-1` for the root itself).
- **`depth[v]`** — edges from the root down to `v`.
- **`size[v]`** — number of nodes in `v`'s subtree (including `v`).

`parent` and `depth` fall out on the way *down*; `size[v] = 1 + Σ size[child]` needs children done first, so we accumulate it on the way back *up* — easiest done by walking the **preorder in reverse** (a child always precedes its parent there). We use an **explicit stack** rather than recursion: a path-graph tree has depth *n*, which would blow Python's recursion limit (see **recursion & backtracking**).

In [1]:
from collections import deque

def root_tree(adj, root=0):
    """One iterative DFS -> parent, depth, subtree size, and a preorder."""
    n = len(adj)
    parent = [-1] * n
    depth  = [0] * n
    order  = []                              # preorder: parents before children
    seen   = [False] * n
    seen[root] = True
    stack = [root]
    while stack:
        u = stack.pop()
        order.append(u)
        for w in adj[u]:
            if not seen[w]:                  # the only "cycle" in a tree is the edge back to parent
                seen[w] = True
                parent[w] = u
                depth[w]  = depth[u] + 1
                stack.append(w)
    size = [1] * n
    for u in reversed(order):                # child precedes parent in reversed preorder
        if parent[u] != -1:
            size[parent[u]] += size[u]
    return parent, depth, size, order

#        0
#       / \
#      1   2
#     /|   |
#    3 4   5
#          |
#          6
adj = [[1, 2], [0, 3, 4], [0, 5], [1], [1], [2, 6], [5]]
parent, depth, size, order = root_tree(adj, root=0)
print("parent:", parent)
print("depth :", depth)
print("size  :", size, " <- size[0] == n, size[2] covers {2,5,6}")

# proofs: edges == n-1, subtree sizes are internally consistent
n = len(adj)
assert sum(len(a) for a in adj) // 2 == n - 1            # a tree has exactly n-1 edges
assert size[0] == n                                      # the root's subtree is everything
for v in range(n):                                       # size = 1 + sum of children's sizes
    kids = [w for w in adj[v] if parent[w] == v]
    assert size[v] == 1 + sum(size[w] for w in kids)
print("checks: n-1 edges, root covers all, size = 1 + sum(children): OK")

parent: [-1, 0, 0, 1, 1, 2, 5]
depth : [0, 1, 1, 2, 2, 2, 3]
size  : [7, 3, 3, 1, 1, 2, 1]  <- size[0] == n, size[2] covers {2,5,6}
checks: n-1 edges, root covers all, size = 1 + sum(children): OK


## 2. Lowest common ancestor — binary lifting

The **lowest common ancestor (LCA)** of two nodes `u` and `v` is the deepest node that is an ancestor of both — informally, where their two root-to-node paths merge. It's the workhorse behind tree distances (`dist(u,v) = depth[u] + depth[v] − 2·depth[lca]`), path queries, and more.

The naive query "walk both pointers up to the root and find the first shared node" is O(n) per query — fine once, painful for many. **Binary lifting** preprocesses a **sparse table** of ancestors so each query drops to **O(log n)**:

> **`up[k][v]` = the 2ᵏ-th ancestor of `v`.**

The table is built by **doubling**: `up[0]` is just `parent`, and `up[k][v] = up[k−1][ up[k−1][v] ]` — the 2ᵏ-th ancestor is the 2ᵏ⁻¹-th ancestor *of* the 2ᵏ⁻¹-th ancestor. That's **O(n log n)** to build. A query then (1) lifts the deeper node up to the shallower one's depth by reading off the **binary digits** of the depth gap, then (2) if they haven't met, lifts both together by the largest jumps that *don't* collide — the node just above is the LCA.

This is exactly the **sparse-table** trick used for range-minimum queries in **fenwick & segment trees**; here the "range" is a path to the root. (The other classic route — flatten the tree by an **Euler tour** and answer LCA as an RMQ over depths — gets O(1) queries after O(n log n) build; binary lifting is simpler and composes naturally with path queries, so we build it from scratch.)

In [2]:
class LCA:
    def __init__(self, adj, root=0):
        n = len(adj)
        self.depth = [0] * n
        parent = [root] * n                  # root's "parent" is itself -> lifts saturate, never index -1
        seen = [False] * n; seen[root] = True
        stack = [root]
        while stack:                         # iterative DFS to set parent + depth
            u = stack.pop()
            for w in adj[u]:
                if not seen[w]:
                    seen[w] = True
                    parent[w] = u
                    self.depth[w] = self.depth[u] + 1
                    stack.append(w)
        self.LOG = max(1, n.bit_length())
        self.up = [[root] * n for _ in range(self.LOG)]
        self.up[0] = parent                          # up[0][v] = 2^0-th ancestor = parent
        for k in range(1, self.LOG):                 # doubling: 2^k-th ancestor = 2^(k-1)-th of the 2^(k-1)-th
            up_prev = self.up[k - 1]
            self.up[k] = [up_prev[up_prev[v]] for v in range(n)]

    def query(self, u, v):
        depth, up = self.depth, self.up
        if depth[u] < depth[v]:
            u, v = v, u                              # make u the deeper one
        diff = depth[u] - depth[v]                   # lift u up by reading the binary digits of the gap
        for k in range(self.LOG):
            if diff >> k & 1:
                u = up[k][u]
        if u == v:                                   # v was an ancestor of u
            return u
        for k in reversed(range(self.LOG)):          # lift both by the biggest jumps that don't collide
            if up[k][u] != up[k][v]:
                u, v = up[k][u], up[k][v]
        return up[0][u]                              # one step up from the split point

# the tree from section 1
adj = [[1, 2], [0, 3, 4], [0, 5], [1], [1], [2, 6], [5]]
lca = LCA(adj, root=0)
print("LCA(3, 4) =", lca.query(3, 4), "(both under 1)")
print("LCA(3, 6) =", lca.query(3, 6), "(paths split at the root)")
print("LCA(6, 2) =", lca.query(6, 2), "(2 is an ancestor of 6)")

LCA(3, 4) = 1 (both under 1)
LCA(3, 6) = 0 (paths split at the root)
LCA(6, 2) = 2 (2 is an ancestor of 6)


Now the empirical proof. We generate hundreds of random trees and random query pairs, and assert binary lifting agrees with the brute-force oracle — *walk one node's ancestors into a set, then walk the other up until it lands in that set*. We also check the distance identity, since LCA's main job is computing tree distances.

In [3]:
import random

def random_tree(n, rng):
    """Random rooted tree on 0..n-1: each node v>=1 attaches to an earlier node."""
    adj = [[] for _ in range(n)]
    for v in range(1, n):
        u = rng.randint(0, v - 1)
        adj[u].append(v); adj[v].append(u)
    return adj

def lca_brute(parent, u, v):
    anc = set()
    while u != -1:                           # collect all ancestors of u
        anc.add(u); u = parent[u]
    while v not in anc:                       # first ancestor of v that's also an ancestor of u
        v = parent[v]
    return v

def bfs_dist(adj, src):                       # ground-truth distances for the identity check
    dist = [-1] * len(adj); dist[src] = 0
    from collections import deque
    q = deque([src])
    while q:
        u = q.popleft()
        for w in adj[u]:
            if dist[w] == -1:
                dist[w] = dist[u] + 1; q.append(w)
    return dist

rng = random.Random(2024)
for _ in range(500):
    n = rng.randint(1, 40)
    adj = random_tree(n, rng)
    parent, depth, _, _ = root_tree(adj, 0)
    lca = LCA(adj, 0)
    for _ in range(15):
        u, v = rng.randrange(n), rng.randrange(n)
        w = lca.query(u, v)
        assert w == lca_brute(parent, u, v)                         # matches brute force
        # distance identity: dist(u,v) = depth[u] + depth[v] - 2*depth[lca]
        assert depth[u] + depth[v] - 2 * depth[w] == bfs_dist(adj, u)[v]
print("binary-lifting LCA == brute force over 500 random trees x 15 queries: OK")
print("distance identity dist = du + dv - 2*d_lca verified too")

binary-lifting LCA == brute force over 500 random trees x 15 queries: OK
distance identity dist = du + dv - 2*d_lca verified too


## 3. Tree diameter — two ways

The **diameter** of a tree is the number of edges on its **longest path** — the two most distant nodes. There are two clean linear-time methods, and it's worth holding both because they generalize differently.

**Method A — two-pass BFS/DFS.** From *any* node, BFS to the farthest node `a`. Then BFS *from `a`* to the farthest node `b`. The path `a–b` is a diameter. The reason this works: the farthest node from anywhere is always *an endpoint of some diameter* (provable on trees, and only on trees — it fails on general graphs). Two linear scans, done.

**Method B — single-DFS DP.** Root the tree anywhere. For each node, let `height[v]` be the longest downward path into its subtree. The longest path that **bends at `v`** (turns the corner through `v`) joins its two **tallest child branches**: `h1 + h2`. Take the max over all nodes. One post-order pass computes every `height` and tracks the best bend — and unlike method A it readily extends to *weighted* edges and to "diameter through a specific node" style queries.

We compute both and assert they agree with each other **and** with a brute-force all-pairs BFS that literally measures every pairwise distance.

In [4]:
from collections import deque

def bfs_farthest(adj, src):
    """Return (farthest node from src, distance array)."""
    dist = [-1] * len(adj); dist[src] = 0
    far = src
    q = deque([src])
    while q:
        u = q.popleft()
        for w in adj[u]:
            if dist[w] == -1:
                dist[w] = dist[u] + 1
                if dist[w] > dist[far]:
                    far = w
                q.append(w)
    return far, dist

def diameter_two_pass(adj):                  # method A
    a, _    = bfs_farthest(adj, 0)           # farthest from an arbitrary start
    b, dist = bfs_farthest(adj, a)           # farthest from a  ->  a..b is a diameter
    return dist[b]

def diameter_dp(adj, root=0):                # method B: single post-order pass
    parent, _, _, order = root_tree(adj, root)
    height = [0] * len(adj)
    best = 0
    for u in reversed(order):                # children settled before parent
        h1 = h2 = 0                          # two tallest downward branches
        for w in adj[u]:
            if w == parent[u]:
                continue
            h = height[w] + 1
            if h > h1:   h1, h2 = h, h1
            elif h > h2: h2 = h
        height[u] = h1
        best = max(best, h1 + h2)            # longest path bending at u
    return best

# a path 0-1-2-3-4 has diameter 4; add a branch and re-measure
path = [[1], [0, 2], [1, 3], [2, 4], [3]]
print("path of 5 nodes  -> two-pass:", diameter_two_pass(path), "| dp:", diameter_dp(path))

adj = [[1, 2], [0, 3, 4], [0, 5], [1], [1], [2, 6], [5]]   # section-1 tree
print("section-1 tree   -> two-pass:", diameter_two_pass(adj), "| dp:", diameter_dp(adj),
      "(e.g. 3-1-0-2-5-6)")

path of 5 nodes  -> two-pass: 4 | dp: 4
section-1 tree   -> two-pass: 5 | dp: 5 (e.g. 3-1-0-2-5-6)


In [5]:
import random

def diameter_brute(adj):                      # oracle: max over all-pairs BFS distances
    best = 0
    for s in range(len(adj)):
        _, dist = bfs_farthest(adj, s)
        best = max(best, max(dist))
    return best

rng = random.Random(99)
for _ in range(500):
    n = rng.randint(1, 45)
    adj = random_tree(n, rng)
    d_a = diameter_two_pass(adj)
    d_b = diameter_dp(adj)
    d_o = diameter_brute(adj)
    assert d_a == d_b == d_o, (n, d_a, d_b, d_o)
print("two-pass BFS == single-DFS DP == brute-force all-pairs over 500 trees: OK")

two-pass BFS == single-DFS DP == brute-force all-pairs over 500 trees: OK


## 4. Rerooting — an answer for every root in O(n)

Some quantities depend on *which node you root at*. The classic is **sum of distances**: `S(v) = Σ dist(v, u)` over all nodes `u`. Computing one `S(v)` is a single BFS, O(n) — but doing it for **every** node the naive way is O(n²). **Rerooting** (a.k.a. *all-roots DP* or *re-rooting technique*) gets the whole array in **O(n)** with two passes.

**Down pass.** Root at node 0 and compute `S(0)` directly: with subtree sizes in hand, `S(0) = Σ depth[v]` (each node contributes its depth). We also have `size[v]` from section 1.

**Reroot pass.** Now slide the root from a node `u` to an adjacent child `c` and ask how `S` changes. Everything *inside* `c`'s subtree gets **one step closer** (there are `size[c]` such nodes); everything *outside* gets **one step farther** (there are `n − size[c]` of them). So:

> **`S(c) = S(u) − size[c] + (n − size[c])`.**

Walk the tree in **preorder** (parent before child) applying that O(1) update, and every `S(v)` is filled in. The same "contribution shifts by ±1 as the root moves" pattern reroots many aggregates — count of nodes at each depth, sum of subtree values, and so on.

In [6]:
def sum_of_distances(adj, root=0):
    """S[v] = sum of distances from v to every node, for ALL v, in O(n)."""
    n = len(adj)
    parent, depth, size, order = root_tree(adj, root)
    S = [0] * n
    S[root] = sum(depth)                     # down pass: anchor the chosen root
    for u in order:                          # reroot in preorder (parent before child)
        for c in adj[u]:
            if c == parent[u]:
                continue
            S[c] = S[u] - size[c] + (n - size[c])    # size[c] closer, n-size[c] farther
    return S

# a star: center is distance 1 to all 4 leaves (S=4); each leaf is 1+2+2+2 = 7
star = [[1, 2, 3, 4], [0], [0], [0], [0]]
print("star S:", sum_of_distances(star), " <- center 4, leaves 7")

adj = [[1, 2], [0, 3, 4], [0, 5], [1], [1], [2, 6], [5]]
print("tree S:", sum_of_distances(adj))

star S: [4, 7, 7, 7, 7]  <- center 4, leaves 7
tree S: [11, 12, 12, 17, 17, 15, 20]


In [7]:
import random

def sum_of_distances_brute(adj):              # O(n^2) oracle: one BFS per node
    out = [0] * len(adj)
    for s in range(len(adj)):
        _, dist = bfs_farthest(adj, s)
        out[s] = sum(dist)
    return out

rng = random.Random(7)
for _ in range(500):
    n = rng.randint(1, 50)
    adj = random_tree(n, rng)
    assert sum_of_distances(adj) == sum_of_distances_brute(adj), n
print("O(n) rerooting == O(n^2) recompute-per-root over 500 random trees: OK")

# and watch the asymptotics actually diverge on one bigger tree
big = random_tree(3000, random.Random(1))
import time
t0 = time.perf_counter(); fast  = sum_of_distances(big);       t1 = time.perf_counter()
t2 = time.perf_counter(); naive = sum_of_distances_brute(big); t3 = time.perf_counter()
assert fast == naive
print(f"n=3000:  reroot {1e3*(t1-t0):6.1f} ms   vs   naive {1e3*(t3-t2):7.1f} ms")

O(n) rerooting == O(n^2) recompute-per-root over 500 random trees: OK


n=3000:  reroot    0.7 ms   vs   naive   755.5 ms


## 5. When to use what

| You need... | Reach for | Time |
|---|---|:---:|
| Parent / depth / subtree size | one rooting DFS (iterative) | O(n) build |
| LCA of two nodes, many queries | **binary lifting** (sparse table) | O(n log n) build, O(log n) query |
| LCA with O(1) queries | **Euler tour + RMQ** (see **fenwick & segment trees**) | O(n log n) build, O(1) query |
| Distance between two nodes | LCA: `du + dv − 2·d_lca` | O(log n) after LCA build |
| Longest path / tree width | **diameter** — two-pass BFS | O(n) |
| Diameter on weighted edges / per-node bend | **diameter** — single-DFS DP | O(n) |
| An aggregate for *every* root | **rerooting** (all-roots DP) | O(n) |
| Path/subtree updates on a flattened tree | Euler tour + **fenwick & segment trees** | O(log n) per op |
| Shortest paths with weights / cycles | it's a **graph** problem (see **graph algorithms**) | varies |

The unifying idea: a tree's missing cycles let one rooting DFS define `parent`, `depth`, and `size` — and LCA, diameter, and rerooting are each a thin layer of bookkeeping over that single pass. When the problem *does* have cycles or weighted shortest paths, you've left tree territory and want the **graph algorithms** notebook.

## 6. Complexity cheat-sheet

| Algorithm | Build / preprocess | Query | Space | Notes |
|---|:---:|:---:|:---:|---|
| Root the tree (parent/depth/size) | O(n) | — | O(n) | iterative DFS; one pass down, one up |
| LCA — binary lifting | O(n log n) | O(log n) | O(n log n) | sparse table `up[k][v]` |
| LCA — Euler tour + sparse-table RMQ | O(n log n) | O(1) | O(n log n) | see **fenwick & segment trees** |
| Tree distance (via LCA) | O(n log n) | O(log n) | O(n log n) | `du + dv − 2·d_lca` |
| Diameter — two-pass BFS/DFS | O(n) | — | O(n) | unweighted; endpoint-of-diameter lemma |
| Diameter — single-DFS DP | O(n) | — | O(n) | `h1 + h2` bend; extends to weights |
| Rerooting (all-roots DP) | O(n) | O(1) per root | O(n) | down pass + ±1 reroot pass |
| Naive per-root recompute | — | O(n) per root | O(n) | O(n²) total — the baseline rerooting beats |

All build costs assume an adjacency-list tree (n nodes, n−1 edges). Everything here used an **iterative** DFS so a degenerate path-tree can't overflow the recursion limit — the same caution the **recursion & backtracking** notebook raises, and the reason the **trees** notebook writes its BST iteratively too.